In [2]:
from tqdm import tqdm
from src import ADP_Dense_Player, Player, comp_models

import json
import os
import logging

DIR_PATH = "_dens2"

RESULTS_PATH = os.path.join(DIR_PATH, "logs/results.log")

# log to file leaderboard_dens2.jsonl
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
fh = logging.FileHandler(RESULTS_PATH)
fh.setLevel(logging.INFO)
logger.addHandler(fh)

def tournament(game_kwargs, models: list[Player], n_test_games: int, start_ind: int = 0) -> list[tuple[int, int, int, int]]:
    total_ind = 0
    with tqdm(
        total=n_test_games * len(models) * (len(models) - 1) // 2, 
        position=0, 
        leave=False, 
        desc="Tournament"
    ) as bar:
        for i in range(len(models)):
            for j in range(i+1, len(models)):
                if total_ind < start_ind:
                    bar.update(n_test_games)
                    total_ind += n_test_games
                    continue
                n_wins, l_history = 0, 0
                for _ in range(n_test_games):
                    win, starts, len_history = comp_models(game_kwargs, models[i], models[j])
                    n_wins += (win > 0) == starts
                    l_history += len_history
                    bar.update(1)
                    total_ind += 1
                n_wins, l_history = n_wins / n_test_games, l_history / n_test_games
                yield (i, j, n_wins, l_history)

game_kwargs = {
    'M': 8,
    'N': 8,
    'K': 5,
    'ADJ': 2,
}

adp_args = {
    "alpha": 0.9,
    "magnify": 1,
    "gamma": 0.9,
    "lr": 0.01,
    "n_steps": 1,
    "epsilon": 0.1,
    "logger": logger, 
    # "M": game_kwargs['M'],
    # "N": game_kwargs['N'],
}

players = [
    ADP_Dense_Player(model_path=os.path.join(DIR_PATH, f"models/epoch_{i}.h5"), **adp_args)
    for i in range(250, 5050, 250)
]

for (i, j, n_wins, l_history) in tournament(game_kwargs, players, n_test_games=25, start_ind=2300):
    logger.info(json.dumps({
        "first": (i + 1) * 250, 
        "second": (j + 1) * 250, 
        "avg_win": n_wins, 
        "avg_len": l_history
    }))

KeyboardInterrupt: 

In [3]:
from src import Gomoku, \
    Player, RandomPlayer, ADP_Player, \
    AlphaZeroPlayer
from IPython.display import clear_output

In [4]:
def my_print(*args, **kwargs):
    clear_output(wait=True)
    print(*args, **kwargs)

def play_game(game: Gomoku, player1: Player, player2: Player, print = None):
    while not game.fin():
        if game.player == 1:
            move = player1.next_move(game)
        else:
            move = player2.next_move(game)
        game.play(move)
        
        if print is not None:
            game.print()
    return game.winner

def play_n_games(game: Gomoku, player1: Player, player2: Player, n: int, print = None):
    avg = 0
    avgs = []
    for i in range(n):
        score = (play_game(game.copy(), player1, player2, print) + 1) / 2
        avg *= i / (i + 1)
        avg += score / (i + 1)
        avgs += [avg]
    return avgs

def time_series_plot(ax, arr: list[float], title: str = ""):
    ax.plot(arr)
    ax.set_title(title)
    ax.set_xlabel("Game")
    ax.set_ylabel("Average score")

In [1]:
game_kwargs = {
    'M': 8,
    'N': 8,
    'K': 5,
    'ADJ': 2,
}

value_network_kwargs = {
    'alpha': 0.9,
    'magnify': 2,
    'gamma': 0.9,
    'lr': 0.01,
    'n_steps': 1,
}

policy_network_kwargs = {
    'epsilon': 0.1,
}

player: Player = None

# policies = {
#     'uct_score': uct_score,
#     'uct_pb_score': uct_pb_score,
#     'pb_score': pb_score,
# }

players = {
    '_RANDOM': RandomPlayer(),
    '_ADP': ADP_Player("models_wzlen/best.h5", value_network_kwargs, policy_network_kwargs),
    '_ALPHAZERO': AlphaZeroPlayer(**game_kwargs),
}

NameError: name 'Player' is not defined

In [4]:
# from tqdm import tqdm
# from matplotlib import pyplot as plt
# n = 25
# scores = {}
# for k in tqdm(range(len(players) ** 2)):
#     player_list, n_players = list(players), len(players)
#     i, j = k // n_players, k % n_players
#     playerI, playerJ = player_list[i], player_list[j]
#     game = Gomoku(
#         M=5,
#         N=5,
#         K=3,
#     )
#     score = play_n_games(game, players[playerI], players[playerJ], n, print=False)
#     scores[(playerI, playerJ)] = score

# fig, axes = plt.subplots(len(players), len(players), figsize=(20, 10))
# player_nums = dict([(v, i) for i, v in enumerate(players)])
# for k, v in scores.items():
#     playerI, playerJ = k
#     ax = axes[player_nums[playerI],  player_nums[playerJ]]
#     time_series_plot(ax, v, f"{playerI} vs {playerJ}")
# plt.show()

In [10]:
game = Gomoku(M=8, N=8, K=5, ADJ=2)

In [7]:
play_game(game, players["_ADP"], players["_ADP"], print=my_print)

Current player: -1
. . . . . . . .
. X . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .

Current player: 1
. . . . . . . .
. X . . . . . .
. . O . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .

Current player: -1
. . . . . . . .
. X . X . . . .
. . O . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .

Current player: 1
. . . . . . . .
. X . X . . . .
. . O . . . . .
. . O . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .

Current player: -1
. . . . . . . .
X X . X . . . .
. . O . . . . .
. . O . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .

Current player: 1
. . . . . . . .
X X . X . . . .
. . O O . . . .
. . O . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .

Current player: -1
. . . . . . . .
X X . X . . . .
. . O O . X . .
. . O . . . . .
. . . . . . . .
. . . . . . . .


-1